# snnTorch Tutorial 6 baseline — walkthrough

Thin walkthrough of the `neuron_baseline` package. The **modules are the truth**; this notebook just imports them, prints what each piece does, runs a very short training run, and renders the plots inline. For a reproducible run use `python -m neuron_baseline train` instead.

See `../README.md` for the full setup + facts pinned from the installed snnTorch source.

In [ ]:
import sys, pathlib
# Make the sibling package importable when running from `notebooks/`.
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 1. Config — the resolved settings that get saved with every run

In [ ]:
from neuron_baseline.config import Config
cfg = Config()
print(cfg.to_json())

## 2. Model — Tutorial 6 verbatim

Conv/pool/Leaky x2 → Flatten → Linear(1024→10) → Leaky(output=True). Every Leaky uses beta=0.5, fast-sigmoid slope=25, threshold=1.0, subtract reset, init_hidden=True.

In [ ]:
from neuron_baseline.model import build_net, forward_pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = build_net(cfg.beta, cfg.surrogate_slope, cfg.threshold, cfg.reset_mechanism).to(device)
net

In [ ]:
# One forward pass on random input, T=8 to stay fast
x = torch.rand(4, 1, 28, 28, device=device)
spk, mem = forward_pass(net, num_steps=8, data=x)
print('spk', tuple(spk.shape), 'mem', tuple(mem.shape))
print('mean spikes per (step, neuron):', float(spk.mean()))

## 3. Data loaders — fixed val split from MNIST train (our adaptation)

In [ ]:
from neuron_baseline.data import build_loaders
loaders = build_loaders(cfg.data_dir, cfg.batch_size, cfg.eval_batch_size,
                        cfg.val_size, cfg.seed)
print('train batches', len(loaders['train']))
print('val examples ',  len(loaders['val'].dataset))
print('test examples', len(loaders['test'].dataset))

## 4. Short training run (2 batches, not a real experiment)

For a real run use `python -m neuron_baseline train` from the shell — the CLI saves config, checkpoints, plots, energy, and the test-set score to `runs/<timestamp>/`. This cell just proves the loop runs end-to-end.

In [ ]:
import itertools, snntorch.functional as SF
from neuron_baseline.reproducibility import seed_everything
seed_everything(cfg.seed)

loss_fn = SF.ce_rate_loss()
optimizer = torch.optim.Adam(net.parameters(), lr=cfg.lr, betas=cfg.adam_betas)

losses, accs = [], []
for data, targets in itertools.islice(loaders['train'], 2):
    data, targets = data.to(device), targets.to(device)
    spk_rec, _ = forward_pass(net, cfg.num_steps, data)
    loss = loss_fn(spk_rec, targets)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(float(loss)); accs.append(float(SF.accuracy_rate(spk_rec, targets)))
print('losses:', [f'{l:.3f}' for l in losses])
print('accs:  ', [f'{a:.3f}' for a in accs])

## 5. Load the artifacts from a real CLI run

After running `python -m neuron_baseline train`, load its `results.json` and inline the plots.

In [ ]:
import json, pathlib
runs = sorted(pathlib.Path('../runs').glob('*'))
print('runs available:', [r.name for r in runs])
if runs:
    latest = runs[-1]
    print('latest:', latest)
    results = json.loads((latest / 'results.json').read_text())
    print('best_val_acc:', results['best_val_accuracy'])
    print('test:', results['test'])
    print('wall_time_s:', results['wall_time_s'])
    print('energy:', json.dumps(results['energy'], indent=2))

In [ ]:
from IPython.display import Image, display
if runs:
    for name in ('loss.png', 'accuracy.png'):
        p = runs[-1] / 'plots' / name
        if p.exists():
            print(p)
            display(Image(filename=str(p)))